# Fast Neural Style Transfer - Training Notebook

This notebook trains a feed-forward neural network to apply artistic style to images in real-time.

**Based on:** [Perceptual Losses for Real-Time Style Transfer](https://arxiv.org/abs/1603.08155) by Johnson et al.

## 1. Setup and Imports

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import requests
from pathlib import Path

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

## 2. Download and Prepare Dataset

In [ ]:
# Download COCO 2017 validation dataset (smaller than train set, ~1GB, 5K images)
# This is a good balance between size and quality for style transfer training

!mkdir -p /kaggle/working/data
!wget -q http://images.cocodataset.org/zips/val2017.zip -O /kaggle/working/data/val2017.zip
!unzip -q /kaggle/working/data/val2017.zip -d /kaggle/working/data/
!rm /kaggle/working/data/val2017.zip

print("Dataset downloaded and extracted!")
print(f"Number of images: {len(os.listdir('/kaggle/working/data/val2017'))}")

In [ ]:
# Download style image (The Great Wave off Kanagawa)
!mkdir -p /kaggle/working/style_images

style_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/0/0a/The_Great_Wave_off_Kanagawa.jpg/1200px-The_Great_Wave_off_Kanagawa.jpg"
style_path = "/kaggle/working/style_images/wave.jpg"

response = requests.get(style_url, headers={"User-Agent": "FastStyleTransfer/1.0"})
with open(style_path, 'wb') as f:
    f.write(response.content)

# Verify the download succeeded
if response.status_code != 200 or len(response.content) < 10000:
    raise RuntimeError(f"Style image download failed (status={response.status_code}, size={len(response.content)} bytes)")

# Display style image
style_img = Image.open(style_path)
plt.figure(figsize=(8, 8))
plt.imshow(style_img)
plt.title("Style Image: The Great Wave")
plt.axis('off')
plt.show()

print(f"Style image downloaded: {style_path}")

## 3. Define Dataset and Transformations

In [ ]:
class ImageDataset(Dataset):
    """Dataset for loading and transforming images."""
    def __init__(self, image_dir, transform=None):
        self.image_dir = Path(image_dir)
        self.image_files = list(self.image_dir.glob('*.jpg'))
        self.transform = transform
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image

# Image transformations
image_size = 256
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
])

# Create dataset and dataloader
dataset = ImageDataset('/kaggle/working/data/val2017', transform=transform)
batch_size = 4
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

print(f"Dataset size: {len(dataset)} images")
print(f"Batch size: {batch_size}")
print(f"Number of batches: {len(dataloader)}")

## 4. Define Transformation Network

In [ ]:
class ResidualBlock(nn.Module):
    """Residual block with two conv layers and skip connection."""
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.in1 = nn.InstanceNorm2d(channels, affine=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.in2 = nn.InstanceNorm2d(channels, affine=True)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        residual = x
        out = self.relu(self.in1(self.conv1(x)))
        out = self.in2(self.conv2(out))
        out = out + residual
        return out

class TransformNet(nn.Module):
    """Image transformation network that learns to apply style."""
    def __init__(self):
        super(TransformNet, self).__init__()
        
        # Downsampling layers (encoder)
        self.conv1 = nn.Conv2d(3, 32, kernel_size=9, padding=4)
        self.in1 = nn.InstanceNorm2d(32, affine=True)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.in2 = nn.InstanceNorm2d(64, affine=True)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.in3 = nn.InstanceNorm2d(128, affine=True)
        
        # Residual blocks
        self.res1 = ResidualBlock(128)
        self.res2 = ResidualBlock(128)
        self.res3 = ResidualBlock(128)
        self.res4 = ResidualBlock(128)
        self.res5 = ResidualBlock(128)
        
        # Upsampling layers (decoder)
        self.deconv1 = nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.in4 = nn.InstanceNorm2d(64, affine=True)
        
        self.deconv2 = nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.in5 = nn.InstanceNorm2d(32, affine=True)
        
        self.conv4 = nn.Conv2d(32, 3, kernel_size=9, padding=4)
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        # Encoder
        x = self.relu(self.in1(self.conv1(x)))
        x = self.relu(self.in2(self.conv2(x)))
        x = self.relu(self.in3(self.conv3(x)))
        
        # Residual blocks
        x = self.res1(x)
        x = self.res2(x)
        x = self.res3(x)
        x = self.res4(x)
        x = self.res5(x)
        
        # Decoder
        x = self.relu(self.in4(self.deconv1(x)))
        x = self.relu(self.in5(self.deconv2(x)))
        x = self.conv4(x)
        
        # Output in range [0, 1]
        x = torch.sigmoid(x)
        
        return x

# Initialize transformation network
transform_net = TransformNet().to(device)
print(f"Transform network parameters: {sum(p.numel() for p in transform_net.parameters()):,}")

## 5. Define VGG16 Loss Network

In [ ]:
class VGG16Features(nn.Module):
    """VGG16 network for extracting features at multiple layers."""
    def __init__(self):
        super(VGG16Features, self).__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        
        # Extract features at specific layers
        self.slice1 = nn.Sequential(*list(vgg.children())[:4])   # relu1_2
        self.slice2 = nn.Sequential(*list(vgg.children())[4:9])  # relu2_2
        self.slice3 = nn.Sequential(*list(vgg.children())[9:16]) # relu3_3
        self.slice4 = nn.Sequential(*list(vgg.children())[16:23]) # relu4_3
        
        # Freeze parameters (we don't train VGG)
        for param in self.parameters():
            param.requires_grad = False
    
    def forward(self, x):
        # Normalize input using ImageNet stats
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(x.device)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(x.device)
        x = (x - mean) / std
        
        h1 = self.slice1(x)
        h2 = self.slice2(h1)
        h3 = self.slice3(h2)
        h4 = self.slice4(h3)
        
        return h1, h2, h3, h4

# Initialize VGG loss network
vgg = VGG16Features().to(device)
vgg.eval()
print("VGG16 loss network loaded")

## 6. Define Loss Functions

In [ ]:
def gram_matrix(features):
    """Compute Gram matrix for style loss."""
    b, c, h, w = features.size()
    features = features.view(b, c, h * w)
    gram = torch.bmm(features, features.transpose(1, 2))
    return gram / (c * h * w)

def content_loss(content_features, generated_features):
    """MSE loss between content and generated features."""
    return F.mse_loss(generated_features, content_features)

def style_loss(style_features, generated_features):
    """MSE loss between Gram matrices of style and generated features."""
    style_gram = gram_matrix(style_features)
    generated_gram = gram_matrix(generated_features)
    
    # Expand style_gram to match batch size of generated_gram
    if style_gram.size(0) == 1 and generated_gram.size(0) > 1:
        style_gram = style_gram.expand_as(generated_gram)
    
    return F.mse_loss(generated_gram, style_gram)

def total_variation_loss(img):
    """Encourages spatial smoothness in the generated image."""
    tv_h = torch.mean(torch.abs(img[:, :, 1:, :] - img[:, :, :-1, :]))
    tv_w = torch.mean(torch.abs(img[:, :, :, 1:] - img[:, :, :, :-1]))
    return tv_h + tv_w

print("Loss functions defined")

## 7. Extract Style Features

In [ ]:
# Load and preprocess style image
style_image = Image.open(style_path).convert('RGB')
style_transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
])
style_tensor = style_transform(style_image).unsqueeze(0).to(device)

# Extract style features once (they don't change during training)
with torch.no_grad():
    style_features = vgg(style_tensor)

print("Style features extracted")
print(f"Style feature shapes: {[f.shape for f in style_features]}")

## 8. Training Setup

In [ ]:
# Hyperparameters
num_epochs = 7
learning_rate = 1e-3
content_weight = 1.0
style_weight = 1e5  # Style loss is typically much higher magnitude
tv_weight = 1e-6

# Optimizer
optimizer = optim.Adam(transform_net.parameters(), lr=learning_rate)

# Create checkpoint directory
checkpoint_dir = Path('/kaggle/working/checkpoints')
checkpoint_dir.mkdir(exist_ok=True)

print(f"Training configuration:")
print(f"  Epochs: {num_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Content weight: {content_weight}")
print(f"  Style weight: {style_weight}")
print(f"  TV weight: {tv_weight}")

## 9. Training Loop

In [ ]:
# Enable multi-GPU if available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    transform_net = nn.DataParallel(transform_net)

transform_net.train()
losses = []
best_loss = float('inf')
best_model_path = checkpoint_dir / "best_model.pth"

for epoch in range(num_epochs):
    epoch_losses = {'total': 0, 'content': 0, 'style': 0, 'tv': 0}
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for batch_idx, content_images in enumerate(pbar):
        content_images = content_images.to(device)
        
        # Generate stylized images
        optimizer.zero_grad()
        stylized_images = transform_net(content_images)
        
        # Extract features
        content_features = vgg(content_images)
        stylized_features = vgg(stylized_images)
        
        # Calculate content loss (use relu3_3, as in the original paper)
        c_loss = content_loss(content_features[2], stylized_features[2])
        
        # Calculate style loss (use all layers)
        s_loss = 0
        for sf, stf in zip(style_features, stylized_features):
            s_loss += style_loss(sf, stf)
        
        # Calculate total variation loss
        tv_loss = total_variation_loss(stylized_images)
        
        # Total loss
        total_loss = (content_weight * c_loss + 
                     style_weight * s_loss + 
                     tv_weight * tv_loss)
        
        # Backpropagation
        total_loss.backward()
        optimizer.step()
        
        # Track losses
        epoch_losses['total'] += total_loss.item()
        epoch_losses['content'] += c_loss.item()
        epoch_losses['style'] += s_loss.item()
        epoch_losses['tv'] += tv_loss.item()
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f"{total_loss.item():.2f}",
            'content': f"{c_loss.item():.4f}",
            'style': f"{s_loss.item():.4f}"
        })
    
    # Average losses for epoch
    for key in epoch_losses:
        epoch_losses[key] /= len(dataloader)
    
    losses.append(epoch_losses)
    
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Total Loss: {epoch_losses['total']:.4f}")
    print(f"  Content Loss: {epoch_losses['content']:.4f}")
    print(f"  Style Loss: {epoch_losses['style']:.4f}")
    print(f"  TV Loss: {epoch_losses['tv']:.6f}")
    
    # Save model only when it's genuinely better (lower total loss)
    if epoch_losses['total'] < best_loss:
        best_loss = epoch_losses['total']
        model_to_save = transform_net.module if isinstance(transform_net, nn.DataParallel) else transform_net
        torch.save(model_to_save.state_dict(), best_model_path)
        print(f"  New best model saved (loss: {best_loss:.4f}): {best_model_path}")
    else:
        print(f"  No improvement (best: {best_loss:.4f}), skipping save")

print("\nTraining completed!")
print(f"Best model loss: {best_loss:.4f}")
print(f"Best model saved at: {best_model_path}")

## 10. Plot Training Losses

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

epochs_range = range(1, num_epochs + 1)

axes[0, 0].plot(epochs_range, [l['total'] for l in losses])
axes[0, 0].set_title('Total Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].grid(True)

axes[0, 1].plot(epochs_range, [l['content'] for l in losses])
axes[0, 1].set_title('Content Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].grid(True)

axes[1, 0].plot(epochs_range, [l['style'] for l in losses])
axes[1, 0].set_title('Style Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].grid(True)

axes[1, 1].plot(epochs_range, [l['tv'] for l in losses])
axes[1, 1].set_title('Total Variation Loss')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_losses.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Test Inference

In [ ]:
# Load the final model
model_to_load = transform_net.module if isinstance(transform_net, nn.DataParallel) else transform_net
model_to_load.eval()

# Get a batch of test images
test_loader = DataLoader(dataset, batch_size=4, shuffle=True)
test_images = next(iter(test_loader)).to(device)

# Generate stylized images
with torch.no_grad():
    stylized_test = model_to_load(test_images)

# Display results
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    # Original
    axes[0, i].imshow(test_images[i].cpu().permute(1, 2, 0))
    axes[0, i].set_title('Original')
    axes[0, i].axis('off')
    
    # Stylized
    axes[1, i].imshow(stylized_test[i].cpu().permute(1, 2, 0))
    axes[1, i].set_title('Stylized')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/test_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Save Final Model

In [ ]:
final_model_path = '/kaggle/working/style_transfer_wave_final.pth'

# Copy best model to final path (best model was already saved during training)
import shutil
shutil.copy(best_model_path, final_model_path)
print(f"Best model copied to: {final_model_path}")
print(f"Best model loss: {best_loss:.4f}")

# Also save model info
model_info = {
    'epochs': num_epochs,
    'image_size': image_size,
    'content_weight': content_weight,
    'style_weight': style_weight,
    'tv_weight': tv_weight,
    'best_loss': best_loss,
    'final_loss': losses[-1]['total']
}

import json
with open('/kaggle/working/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print("\nModel Info:")
for key, value in model_info.items():
    print(f"  {key}: {value}")

## 13. Inference Function (for later use)

In [ ]:
def stylize_image(image_path, model_path, output_path=None):
    """
    Apply style transfer to a single image.
    
    Args:
        image_path: Path to input image
        model_path: Path to trained model
        output_path: Optional path to save output (if None, just displays)
    """
    # Load model
    model = TransformNet().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    # Generate stylized image
    with torch.no_grad():
        stylized = model(image_tensor)
    
    # Convert to PIL image
    stylized = stylized.cpu().squeeze(0).permute(1, 2, 0).numpy()
    stylized = (stylized * 255).astype(np.uint8)
    stylized_image = Image.fromarray(stylized)
    
    if output_path:
        stylized_image.save(output_path)
        print(f"Stylized image saved to: {output_path}")
    
    # Display
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(image)
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(stylized_image)
    axes[1].set_title('Stylized')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    
    return stylized_image

# Example usage:
# stylized = stylize_image('path/to/image.jpg', final_model_path, 'output.jpg')

---

## Training Complete!

**Next Steps:**
1. Use the `stylize_image()` function to apply style to new images
2. Experiment with different style images by changing the download URL
3. Tune hyperparameters (content_weight, style_weight) for different effects
4. Train for more epochs for better quality

**Files Generated:**
- `style_transfer_wave_final.pth` - Final trained model
- `model_info.json` - Training configuration
- `checkpoints/` - Training checkpoints
- `training_losses.png` - Loss curves
- `test_results.png` - Sample outputs